# Notebook 2: VideoMAE Fine-Tuning

**Account:** B | **GPU:** T4 x2 | **Time:** ~12-15 hrs

**Datasets to attach:** `daisee-videos` + `smartlms-scripts` (or whatever you named them)

Fine-tunes VideoMAE V2 end-to-end on raw DAiSEE video clips — all 4 dimensions.
Then extracts learned features for the multi-modal fusion step.

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q "numpy<2" transformers accelerate decord av xgboost optuna

import torch, os, sys, shutil, glob, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"Multi-GPU: {torch.cuda.device_count()} GPUs — DataParallel enabled")

In [ ]:
# ── Cell 2: Copy scripts + setup ──
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/app/ml", exist_ok=True)
open(f"{WORK}/app/__init__.py", "w").close()
open(f"{WORK}/app/ml/__init__.py", "w").close()

def find_script(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

scripts = [
    "train_model_v2.py", "train_model_v3.py", "train_model_v4.py",
    "train_multimodal_v5.py", "extract_face_embeddings.py",
    "train_videomae.py", "extract_audio_features.py"
]
for s in scripts:
    found = find_script(s)
    if found:
        shutil.copy(found, f"{WORK}/app/ml/{s}")
        print(f"✓ {s} ← {found}")
    else:
        print(f"✗ NOT FOUND: {s}")

sys.path.insert(0, WORK)
os.chdir(WORK)

In [ ]:
# ── Cell 3: Auto-detect & patch paths ──
print("Attached datasets:")
for d in sorted(os.listdir("/kaggle/input")):
    print(f"  /kaggle/input/{d}/")

# Find video DataSet directory
result = subprocess.run(["find", "/kaggle/input", "-name", "DataSet", "-type", "d"], capture_output=True, text=True)
dataset_dirs = [l for l in result.stdout.strip().split("\n") if l]
VIDEO_DIR = dataset_dirs[0] if dataset_dirs else None
print(f"\nVideo DataSet dir: {VIDEO_DIR}")

# Find Labels
result = subprocess.run(["find", "/kaggle/input", "-name", "AllLabels.csv"], capture_output=True, text=True)
label_hits = [l for l in result.stdout.strip().split("\n") if l]
LABELS_DIR = os.path.dirname(label_hits[0]) if label_hits else None
DAISEE_DIR = os.path.dirname(LABELS_DIR) if LABELS_DIR else None
print(f"Labels dir: {LABELS_DIR}")
print(f"DAiSEE dir: {DAISEE_DIR}")

# Count videos
if VIDEO_DIR:
    n_avi = len(glob.glob(os.path.join(VIDEO_DIR, "**", "*.avi"), recursive=True))
    print(f"\nFound {n_avi} .avi video clips")

# Patch train_videomae.py
vmae_path = f"{WORK}/app/ml/train_videomae.py"
code = open(vmae_path).read()
if VIDEO_DIR:
    code = code.replace(
        'VIDEO_DIR = "/kaggle/input/daisee-videos/DAiSEE/DataSet"',
        f'VIDEO_DIR = "{VIDEO_DIR}"'
    )
if LABELS_DIR:
    code = code.replace(
        'LABELS_DIR = "/kaggle/input/smartlms-openface/Labels"',
        f'LABELS_DIR = "{LABELS_DIR}"'
    )
open(vmae_path, 'w').write(code)
print("\n✓ Patched train_videomae.py paths")

print(f"\n✅ Expected: ~8231 video clips in DataSet/")

In [ ]:
# ── Cell 4: Fine-tune VideoMAE on ALL 4 dimensions (~12 hrs) ──
# Each dimension = ~3 hrs on T4 with gradient checkpointing
# Models saved after each dimension, so safe to resume
from app.ml.train_videomae import train_videomae, DIMENSION_NAMES, RESULTS_DIR
import json, torch, gc
from datetime import datetime

results = {}
for dim in DIMENSION_NAMES:
    print(f"\n{'='*60}")
    print(f"Fine-tuning VideoMAE for: {dim}")
    print(f"{'='*60}")
    result = train_videomae(
        dim_name=dim,
        num_frames=16,
        epochs=30,
        batch_size=4,
        lr=2e-5,
        binary=True,
        model_name="MCG-NJU/videomae-base",
    )
    results[dim] = result
    # Free GPU memory between dimensions
    gc.collect()
    torch.cuda.empty_cache()

# Save results (same as main() would do)
os.makedirs(RESULTS_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(RESULTS_DIR, f"experiment_videomae_{timestamp}.json")
with open(out_path, 'w') as f:
    json.dump({
        "timestamp": datetime.now().isoformat(),
        "model": "MCG-NJU/videomae-base",
        "mode": "finetune_all",
        "results": results,
    }, f, indent=2, default=str)
print(f"\n✅ VideoMAE fine-tuning complete! Results: {out_path}")

In [ ]:
# ── Cell 5: Extract VideoMAE learned features (~2 hrs) ──
# These features get used in the multi-modal fusion (Notebook 5)
from app.ml.train_videomae import extract_videomae_features
import torch, gc

extract_videomae_features(
    model_name="MCG-NJU/videomae-base",
    num_frames=16,
    batch_size=8,
)

gc.collect()
torch.cuda.empty_cache()
print("\n✅ VideoMAE feature extraction complete!")

In [ ]:
# ── Cell 6: Review results ──
import json

print("=" * 80)
print("VIDEOMAE RESULTS")
print("=" * 80)

for rf in sorted(glob.glob("/kaggle/working/experiment_results/*videomae*.json")):
    print(f"\n{os.path.basename(rf)}")
    with open(rf) as f:
        data = json.load(f)
    for key, val in data.items():
        if isinstance(val, dict) and 'test_f1_macro' in val:
            print(f"  {key}: F1m = {val['test_f1_macro']:.4f}")
        elif isinstance(val, (int, float)):
            print(f"  {key}: {val}")

# Check extracted features
feat_dir = "/kaggle/working/trained_models/videomae_features"
if os.path.exists(feat_dir):
    feats = glob.glob(os.path.join(feat_dir, "*.npy"))
    print(f"\nExtracted features: {len(feats)} files")
else:
    print("\n⚠ No extracted features found")

In [ ]:
# ── Cell 7: Save outputs (models + features) ──
shutil.make_archive("/kaggle/working/videomae_models", 'zip', "/kaggle/working/trained_models")
shutil.make_archive("/kaggle/working/videomae_results", 'zip', "/kaggle/working/experiment_results")

for f in glob.glob("/kaggle/working/*.zip"):
    size_mb = os.path.getsize(f) / 1e6
    print(f"{os.path.basename(f)}: {size_mb:.1f} MB")

print("\n✅ Download from Output tab!")
print("The videomae_features/ + trained_models/ will be needed for Notebook 5 (fusion).")
print("TIP: Re-upload the 'videomae_features' folder as a new Kaggle dataset for Notebook 5.")